In [3]:
import pandas as pd

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

df.head()

Dataset shape: (7043, 21)

Column names:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
# TotalCharges mein blank strings ko numeric missing values banayein
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"], errors="coerce"
)

# Target: No = 0, Yes = 1
y = df["Churn"].map({"No": 0, "Yes": 1})

# ID aur target ko predictors se hata dein
X = df.drop(
    columns=["Churn", "customerID"],
    errors="ignore"
)

print("Missing TotalCharges:", df["TotalCharges"].isna().sum())
print("\nChurn distribution:")
print(y.value_counts())
print("\nChurn percentage:")
print((y.value_counts(normalize=True) * 100).round(2))

Missing TotalCharges: 11

Churn distribution:
Churn
0    5174
1    1869
Name: count, dtype: int64

Churn percentage:
Churn
0    73.46
1    26.54
Name: proportion, dtype: float64


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print("\nTraining churn rate:", round(y_train.mean(), 3))
print("Testing churn rate:", round(y_test.mean(), 3))

Training rows: 5634
Testing rows: 1409

Training churn rate: 0.265
Testing churn rate: 0.265


In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

numeric_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    exclude=["number"]
).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

print("Numeric columns:", numeric_features)
print("Categorical columns:", categorical_features)

Numeric columns: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
Categorical columns: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [7]:
model_default = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=2000, random_state=42))
])

model_balanced = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

model_default.fit(X_train, y_train)
model_balanced.fit(X_train, y_train)

print("Dono models successfully train ho gaye!")

Dono models successfully train ho gaye!


In [8]:
import numpy as np

majority_class = int(y_train.mode().iloc[0])

baseline_predictions = np.full(
    len(y_test), majority_class
)

print("Majority class:", majority_class)
print("0 = No churn, 1 = Churn")

Majority class: 0
0 = No churn, 1 = Churn


In [9]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

def evaluate_model(name, y_true, predictions, probabilities):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, predictions),
        "Precision": precision_score(
            y_true, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_true, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_true, predictions, zero_division=0
        ),
        "ROC-AUC": roc_auc_score(y_true, probabilities)
    }

pred_default = model_default.predict(X_test)
pred_balanced = model_balanced.predict(X_test)

prob_default = model_default.predict_proba(X_test)[:, 1]
prob_balanced = model_balanced.predict_proba(X_test)[:, 1]

# Constant majority-class score ka ROC-AUC 0.5 hota hai
baseline_scores = np.full(len(y_test), majority_class)

results = pd.DataFrame([
    evaluate_model(
        "Majority-class baseline",
        y_test, baseline_predictions, baseline_scores
    ),
    evaluate_model(
        "Logistic Regression - Default",
        y_test, pred_default, prob_default
    ),
    evaluate_model(
        "Logistic Regression - Balanced",
        y_test, pred_balanced, prob_balanced
    )
])

results = results.set_index("Model").round(3)
results

,Accuracy,Precision,Recall,F1,ROC-AUC
Model,,,,,
Majority-class baseline,0.735,0.000,0.000,0.000,0.500
Logistic Regression - Default,0.806,0.657,0.559,0.604,0.842
Logistic Regression - Balanced,0.738,0.504,0.783,0.614,0.841


In [10]:
from sklearn.metrics import confusion_matrix, classification_report

models_predictions = {
    "Majority Baseline": baseline_predictions,
    "Logistic Regression - Default": pred_default,
    "Logistic Regression - Balanced": pred_balanced
}

for name, predictions in models_predictions.items():
    print("\n", "=" * 50)
    print(name)
    print("Confusion matrix:")
    print(confusion_matrix(y_test, predictions, labels=[0, 1]))
    print("\nClassification report:")
    print(classification_report(
        y_test,
        predictions,
        labels=[0, 1],
        target_names=["No Churn", "Churn"],
        zero_division=0
    ))


Majority Baseline
Confusion matrix:
[[1035    0]
 [ 374    0]]

Classification report:
              precision    recall  f1-score   support

    No Churn       0.73      1.00      0.85      1035
       Churn       0.00      0.00      0.00       374

    accuracy                           0.73      1409
   macro avg       0.37      0.50      0.42      1409
weighted avg       0.54      0.73      0.62      1409


Logistic Regression - Default
Confusion matrix:
[[926 109]
 [165 209]]

Classification report:
              precision    recall  f1-score   support

    No Churn       0.85      0.89      0.87      1035
       Churn       0.66      0.56      0.60       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409


Logistic Regression - Balanced
Confusion matrix:
[[747 288]
 [ 81 293]]

Classification report:
              precision    recall  f1-score   support

    No Chu

In [11]:
default_recall = results.loc[
    "Logistic Regression - Default", "Recall"
]
balanced_recall = results.loc[
    "Logistic Regression - Balanced", "Recall"
]

default_precision = results.loc[
    "Logistic Regression - Default", "Precision"
]
balanced_precision = results.loc[
    "Logistic Regression - Balanced", "Precision"
]

print(f"Default model recall: {default_recall:.3f}")
print(f"Balanced model recall: {balanced_recall:.3f}")
print(f"Default model precision: {default_precision:.3f}")
print(f"Balanced model precision: {balanced_precision:.3f}")

if balanced_recall > default_recall:
    print("\nBalanced model catches more churners on this test set.")
elif balanced_recall < default_recall:
    print("\nDefault model has higher churn recall on this test set.")
else:
    print("\nBoth models have the same churn recall on this test set.")

Default model recall: 0.559
Balanced model recall: 0.783
Default model precision: 0.657
Balanced model precision: 0.504

Balanced model catches more churners on this test set.


In [12]:
results.to_csv("model_metrics.csv")

with open("MODEL_COMPARISON.md", "w", encoding="utf-8") as f:
    f.write("# Telco Customer Churn — Model Comparison\n\n")

    f.write("## Test-set metrics\n\n")
    f.write(results.to_markdown())
    f.write("\n\n## Confusion matrices\n\n")

    for name, predictions in models_predictions.items():
        cm = confusion_matrix(y_test, predictions, labels=[0, 1])
        f.write(f"### {name}\n\n")
        f.write("Rows = actual; columns = predicted; order = No, Yes.\n\n")
        f.write("| | Predicted No | Predicted Yes |\n")
        f.write("|---|---:|---:|\n")
        f.write(f"| Actual No | {cm[0,0]} (TN) | {cm[0,1]} (FP) |\n")
        f.write(f"| Actual Yes | {cm[1,0]} (FN) | {cm[1,1]} (TP) |\n\n")

    f.write("## Business interpretation\n\n")
    f.write(
        "Compare churn recall, precision, F1 and false negatives. "
        "If missed churners are more costly than false alarms, prefer "
        "the model that catches more churners when its outreach burden "
        "is acceptable. ROC-AUC evaluates ranking across thresholds; "
        "it does not by itself determine the best business decision. "
        "The models here use the default 0.50 classification threshold.\n"
    )

print("Report files created!")

Report files created!


In [13]:
from google.colab import files

files.download("MODEL_COMPARISON.md")
files.download("model_metrics.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>